In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

115

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.

Return ONLY a valid JSON object with a "questions" key containing a list of 5 strings.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()
openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url ="https://api.groq.com/openai/v1"
)

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "system", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    response_format={"type": "json_object"},
    max_completion_tokens=2000,
    extra_body={"reasoning_format": "hidden", "reasoning_effort": "none"}
)

In [11]:
content = response.choices[0].message.content
data = json.loads(content)
result = Questions(**data)
print(result)
print(result.questions)

questions=['Is it too late to sign up for the LLM Zoomcamp since I just found it?', 'Can I still get a certificate if I join late?', 'What are the requirements to get my certificate if I start now?', 'Does joining late affect my eligibility for the project submission?', "I'm a bit behind schedule, can I still participate and get certified?"]
['Is it too late to sign up for the LLM Zoomcamp since I just found it?', 'Can I still get a certificate if I join late?', 'What are the requirements to get my certificate if I start now?', 'Does joining late affect my eligibility for the project submission?', "I'm a bit behind schedule, can I still participate and get certified?"]


In [12]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found out about this LLM course, is it still possible for me to sign up and participate?', 'If I start the course now, am I eligible to get the certificate at the end?', 'What are the requirements to receive a certificate if I join the course late?', 'Does the certificate bonus apply to students who join after the course has already started?', 'I missed the start date, but I still want the certificate. Do I just need to submit the project on time?']


In [13]:
from evaluation_utils import calc_price

# Groq uses prompt_tokens/completion_tokens not input_tokens/output_tokens
print(usage.prompt_tokens, usage.completion_tokens)

cost = calc_price(usage)
print(cost)

213 123
{'input_cost': 0.0001278, 'output_cost': 0.000369, 'total_cost': 0.0004968}


In [14]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

In [22]:
import json
import time
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            'question': q,
            'documents': doc['id']
        })
    
    # Corrected to use the time module prefix
    time.sleep(2)
    
    return results, usage

In [23]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [24]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# Lowering max_workers to 2 avoids hitting the rate limit wall immediately
with ThreadPoolExecutor(max_workers=1) as pool:
     results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/115 [00:00<?, ?it/s]

Rate limit or API error encountered. Retrying in 15 seconds... (Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.6-27b` in organization `org_01kv3ef832f5cv2g6vbabmng1d` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199600, Requested 701. Please try again in 2m10.032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Rate limit or API error encountered. Retrying in 30 seconds... (Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.6-27b` in organization `org_01kv3ef832f5cv2g6vbabmng1d` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199565, Requested 685. Please try again in 1m48s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Rate limit or API error encountered. Retrying in 15 seco

KeyboardInterrupt: 

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)